# Sentiment Analysis of Movie Reviews Using LSTM (Deep Learning Model)

**Student name:** [your name]  |  **Roll number:** [your roll number]  |  **Course:** [course]  |  **Instructor:** [instructor]  |  **Date:** [date]

> How to use this notebook: run the cells from top to bottom. Every step has the same layout:
> **What** we do, **Why** it is needed, the **code**, a **simple explanation**, the **expected output**, and **common errors**.
> No results are pre-filled. Every number and graph is produced when *you* run the cells.

---
# Step 1: Project Introduction

## 1.1 Assignment title
Sentiment Analysis of Movie Reviews Using LSTM (Deep Learning Model)

## 1.2 Introduction to sentiment analysis
**Sentiment analysis** (also called opinion mining) is the task of finding the feeling or opinion in a piece of text.
For example, "A brilliant, moving film" expresses a *positive* opinion, while "Dull and far too long" expresses a *negative* one.
Companies use it to study customer reviews, social media posts and survey answers without reading thousands of messages by hand.

Sentiment analysis is a **text classification** problem: the input is text and the output is a category (here, Positive or Negative).

## 1.3 Problem statement
Given the text of a movie review written by a person, build a deep learning model that automatically predicts whether the review is
**Positive** or **Negative**, and that still works on new reviews it has never seen before.

## 1.4 Assignment objectives
1. Load and explore the IMDB movie review dataset.
2. Convert text into numbers a neural network can use (tokenized sequences, padding).
3. Build an LSTM neural network for binary classification.
4. Train the model and watch for overfitting using a validation set.
5. Evaluate the model with accuracy, precision, recall, F1-score and a confusion matrix.
6. Use the trained model to predict the sentiment of new reviews.
7. Document the work and publish it on GitHub.

## 1.5 What is an LSTM?
A **neural network** is a model made of layers of small calculation units. During training it adjusts its internal numbers (weights) so its
predictions match the correct answers more often.

A **Recurrent Neural Network (RNN)** reads a sequence one item at a time (for text, one word at a time) and passes a *memory* (hidden state) forward.
Ordinary RNNs struggle to remember information from many steps ago (the "vanishing gradient" problem).

An **LSTM (Long Short-Term Memory)** network is a special RNN (Hochreiter & Schmidhuber, 1997) that has a **cell state**, a kind of conveyor belt of memory,
controlled by three **gates**:

| Gate | Job (simple English) |
|---|---|
| Forget gate | Decides which old information to throw away |
| Input gate | Decides which new information to store |
| Output gate | Decides what part of the memory to pass on as output |

```
   word 1 --> [LSTM cell] --> word 2 --> [LSTM cell] --> ... --> word n --> [LSTM cell] --> final summary --> Dense --> Positive/Negative
                  |                          |                                    |
              memory carried forward ----------------------------------------------
```

## 1.6 Why is LSTM suitable for sentiment classification?
- **Word order matters.** "Not good" and "good" have opposite meanings. LSTMs read words in order.
- **Context can be far away.** In "The beginning was slow and confusing, but the ending was absolutely superb", the final opinion depends on words far apart.
  Gates let the LSTM keep important earlier information.
- **Variable length.** Reviews can be short or long; LSTMs process sequences of any length (we pad them to a fixed size only for batching).
- **Learns by itself.** The network learns useful word representations (embeddings) and patterns from the data without hand-written rules.

(Newer models such as Transformers are stronger on many tasks, but LSTM is an excellent, understandable model for learning deep learning for text.)

## 1.7 Expected outcome
- A trained LSTM model that classifies a review as Positive or Negative with a confidence score.
- Training curves, a confusion matrix and metrics (accuracy, precision, recall, F1) computed on 25,000 unseen test reviews.
- Typically, a small LSTM like the one here reaches a test accuracy in the mid-80% range on IMDB. This is a general expectation, **not a guarantee**.
  Your own numbers will depend on the settings and the random seed, and you should report only what your run prints.

---
# Step 2: Environment Setup

**What:** install Python, create an isolated environment and install the libraries.
**Why:** a virtual environment keeps this project's library versions separate from other projects, which prevents version conflicts.

## Option A: Windows 11 with VS Code

**1. Check Python** (use Python **3.10, 3.11 or 3.12**, 64-bit). Open VS Code, then *Terminal > New Terminal* (PowerShell):
```powershell
python --version
pip --version
```
If `python` is not recognized, install it from python.org and tick **"Add python.exe to PATH"**, then reopen VS Code.

**2. Create a project folder and open it in VS Code**
```powershell
mkdir lstm-sentiment-analysis
cd lstm-sentiment-analysis
code .
```

**3. Create the virtual environment** (creates a private `.venv` folder)
```powershell
python -m venv .venv
```

**4. Activate it**
```powershell
.\.venv\Scripts\Activate.ps1
```
Your prompt should now start with `(.venv)`. If PowerShell says *"running scripts is disabled"*, run this once and activate again:
```powershell
Set-ExecutionPolicy -Scope CurrentUser -ExecutionPolicy RemoteSigned
```
(Alternative without changing policy: use Command Prompt and run `.venv\Scripts\activate.bat`.)

**5. Install the libraries** (TensorFlow is a large download)
```powershell
python -m pip install --upgrade pip
pip install -r requirements.txt
```
The `requirements.txt` file is included with this project. To install manually instead:
```powershell
pip install tensorflow numpy pandas matplotlib seaborn scikit-learn jupyter ipykernel
```

**6. Install the VS Code extensions** *Python* and *Jupyter* (Extensions icon on the left bar). Open this notebook (`.ipynb`), click **Select Kernel**
(top right) and choose the Python interpreter inside `.venv`.

**7. Run the project.** Click *Run All*, or run each cell with `Shift + Enter`.
Optionally, from the terminal: `jupyter notebook` opens the notebook in your browser.

> Native Windows TensorFlow runs on the **CPU** only (GPU support ended after TensorFlow 2.10). This model is small, so a CPU is fine, but training takes a few minutes.

## Option B: Google Colab (no installation)
1. Go to colab.research.google.com, sign in, choose *File > Upload notebook* and select this `.ipynb` file.
2. *Optional speed-up:* *Runtime > Change runtime type > T4 GPU*.
3. TensorFlow, NumPy, Pandas, Matplotlib, Seaborn and Scikit-learn are already installed in Colab, so you can run the cells directly.
4. Colab sessions are temporary. Files saved by the notebook (the model, the plots) disappear when the session ends, so download what you need
   (*Files* panel on the left, or `from google.colab import files; files.download("lstm_sentiment.keras")`).

## Common errors in this step
| Error | Fix |
|---|---|
| `'python' is not recognized` | Reinstall Python with "Add to PATH" ticked, reopen VS Code |
| `Activate.ps1 cannot be loaded` | Run the `Set-ExecutionPolicy` command above |
| `No matching distribution found for tensorflow` | Python too new/old or 32-bit. Install 64-bit Python 3.10-3.12 and recreate `.venv` |
| Notebook says `ModuleNotFoundError` | The wrong kernel is selected. Pick the `.venv` interpreter and restart the kernel |
| `DLL load failed` when importing TensorFlow | Install the Microsoft Visual C++ Redistributable (x64), restart |
| Messages about oneDNN / CPU instructions | Informational, not errors |

In [2]:
# Check that the environment works
import sys, platform
import tensorflow as tf
print("Python     :", sys.version.split()[0], "|", platform.system())
print("TensorFlow :", tf.__version__)
print("GPU found  :", tf.config.list_physical_devices("GPU") or "none (CPU will be used)")


[TensorFlow DLL Diagnostic] Analyzing: c:\Users\DELL\Downloads\lstm-sentiment-analysis\.venv\lib\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd
[Error] Failed to load _pywrap_tensorflow_common.dll: INITIALIZATION FAILED (0x45A) - The DLL's DllMain returned false.
    Hint: This often happens if your CPU lacks required instructions (like AVX/AVX2)
    or if the Microsoft Visual C++ Redistributable is outdated/missing.


ImportError: Traceback (most recent call last):
  File "c:\Users\DELL\Downloads\lstm-sentiment-analysis\.venv\lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 74, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

**Code explained.** `sys.version` prints your Python version, `tf.__version__` proves TensorFlow imported correctly, and the last line lists any GPU.
**Expected output:** three lines, e.g. `Python : 3.11.x | Windows`, a TensorFlow version number, and `GPU found : none (CPU will be used)` on a normal Windows PC.
**Common errors:** `ModuleNotFoundError: No module named 'tensorflow'` means the installation (Step 2) did not finish or the wrong kernel is selected.

---
# Step 3: Import Libraries and Set Up Reproducibility

**What:** load all the tools we need and fix the random seeds.
**Why:** neural networks start from random weights. Setting a seed makes your results repeatable (as far as the hardware allows), which matters for a report.

In [ ]:
import random, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

sns.set_theme(style="whitegrid")
print("Libraries imported.")

**Code explained.** NumPy and Pandas handle numbers and tables; Matplotlib and Seaborn draw graphs; TensorFlow/Keras builds the network;
Scikit-learn provides the evaluation metrics and the train/validation split. `set_random_seed` fixes the random numbers.
**Expected output:** `Libraries imported.`
**Common errors:** `ModuleNotFoundError: No module named 'seaborn'` (or another library) means it is not installed in the active environment: run `pip install seaborn` and restart the kernel.

---
# Step 4: Settings and Loading the IMDB Dataset

**What:** define the main settings in one place, then load the IMDB dataset that is built into Keras.
**Why:** the model needs labelled examples to learn from. IMDB has **50,000** movie reviews, split into **25,000 for training** and **25,000 for testing**,
with exactly half positive and half negative in each part. The test reviews are kept untouched so we can judge the model fairly at the end.

Key settings:
- `VOCAB_SIZE = 10000`: only the 10,000 most frequent words are kept; rarer words become an "unknown" marker. This keeps the model small.
- `MAX_LEN = 200`: every review will be cut or padded to 200 words so reviews can be processed in batches.

In [ ]:
VOCAB_SIZE = 10000     # number of most frequent words to keep
MAX_LEN = 200          # every review becomes exactly this many words
EMBEDDING_DIM = 64     # size of the vector that represents each word
LSTM_UNITS = 64        # memory size of the LSTM
BATCH_SIZE = 64
EPOCHS = 8             # maximum passes over the training data
LABELS = {0: "Negative", 1: "Positive"}

# The first run downloads the data (about 17 MB) and stores it in ~/.keras/datasets
(x_train_raw, y_train), (x_test_raw, y_test) = keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)

print("Training reviews:", len(x_train_raw))
print("Test reviews    :", len(x_test_raw))
print("First training review (as word numbers), first 15 numbers:", x_train_raw[0][:15])
print("Its label:", y_train[0], "->", LABELS[int(y_train[0])])

**Code explained.** `imdb.load_data` returns two pairs: (training reviews, training labels) and (test reviews, test labels). Each review is a **list of integers**;
every integer stands for one word (the dataset was already converted to numbers for us). Labels are `0` (negative) or `1` (positive).
**Expected output:** `Training reviews: 25000`, `Test reviews: 25000`, a list of integers starting with `1` (the special "start of review" marker), and a label of 0 or 1.
**Common errors:**
| Error | Fix |
|---|---|
| `URLError` / download failed / `Exception: URL fetch failure` | Check your internet, VPN or proxy and rerun the cell |
| Corrupted download (`EOFError`, `BadZipFile`) | Delete the file `imdb.npz` from `C:\Users\<you>\.keras\datasets\` and rerun |
| Very slow first run | Normal: it is downloading |

---
# Step 5: Explore the Data

**What:** look at the class balance, the review lengths, and a review converted back into words.
**Why:** understanding data prevents mistakes. We need to know whether the classes are balanced (this affects which metrics are meaningful) and how long reviews are
(this lets us choose a sensible `MAX_LEN`).

Keras reserves the first numbers: `0` = padding, `1` = start of review, `2` = unknown word, `3` = unused. Real words are numbered from `4`,
so a word that has number *n* in the official dictionary appears in the reviews as *n + 3*.

In [ ]:
# 5a. Class balance
train_labels = pd.Series(y_train).map(LABELS)
print(train_labels.value_counts())

# 5b. Review lengths (number of words per review)
lengths = pd.Series([len(review) for review in x_train_raw])
print("\nReview length statistics (words):")
print(lengths.describe().round(1))
print(f"\nShare of reviews longer than MAX_LEN ({MAX_LEN}): {(lengths > MAX_LEN).mean():.1%}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x=train_labels, ax=ax[0], palette=["#d9534f", "#5cb85c"], hue=train_labels, legend=False)
ax[0].set_title("Class balance (training set)"); ax[0].set_xlabel("Sentiment")
sns.histplot(lengths, bins=50, ax=ax[1], color="#4C72B0")
ax[1].axvline(MAX_LEN, color="red", linestyle="--", label=f"MAX_LEN = {MAX_LEN}")
ax[1].set_title("Review length distribution"); ax[1].set_xlabel("Words per review"); ax[1].legend()
plt.tight_layout(); plt.show()

# 5c. Turn one review back into readable text
word_index = keras.datasets.imdb.get_word_index()                      # word -> number
index_to_word = {number + 3: word for word, number in word_index.items()}
index_to_word.update({0: "<PAD>", 1: "<START>", 2: "<UNK>", 3: "<UNUSED>"})

def decode_review(numbers):
    return " ".join(index_to_word.get(n, "<UNK>") for n in numbers)

print("\nSample review (decoded):")
print(decode_review(x_train_raw[1])[:500], "...")
print("Label:", LABELS[int(y_train[1])])

**Code explained.** `value_counts` counts how many reviews are in each class. `describe()` summarizes review lengths. The bar chart shows class balance and the histogram
shows lengths with a red line at `MAX_LEN`. `decode_review` reverses the number dictionary so we can read a review as text. Unknown words appear as `<UNK>`.
**Expected output:** a count of 12,500 for each class (the dataset is perfectly balanced), a table of length statistics, the share of reviews longer than 200 words,
two charts, and a readable (lower-case, punctuation-free) review text with its label. The length distribution is skewed: most reviews are fairly short and a few are very long.
**Because the classes are balanced,** accuracy is a fair headline metric (we still report precision, recall and F1).
**Common errors:** `ValueError` with `palette` in an old Seaborn version: upgrade (`pip install -U seaborn`) or delete the `palette=` argument.

---
# Step 6: Preprocess the Data (Padding and Splitting)

**What:** make all reviews the same length and create a validation set.
**Why:**
- Neural networks process data in rectangular batches, so every review must have the same length. Long reviews are **truncated** and short ones are **padded** with zeros.
- We hold out **20% of the training data as a validation set**. The model never learns from it, so it tells us during training whether the model is generalizing or just memorizing (overfitting).
  The 25,000 **test** reviews stay completely untouched until the final evaluation.

In [ ]:
def pad_reviews(sequences):
    return pad_sequences(sequences, maxlen=MAX_LEN, padding="post", truncating="post", value=0)

x_train_all = pad_reviews(x_train_raw)
x_test = pad_reviews(x_test_raw)

# 80% training / 20% validation, keeping the same class balance in both
x_train, x_val, y_train_split, y_val = train_test_split(
    x_train_all, y_train, test_size=0.2, stratify=y_train, random_state=SEED)

print("Training set  :", x_train.shape)
print("Validation set:", x_val.shape)
print("Test set      :", x_test.shape)
print("\nFirst review after padding (first 12 numbers):", x_train[0][:12])
print("Last 8 numbers (zeros = padding):", x_train[0][-8:])

**Code explained.** `pad_sequences` forces each review to `MAX_LEN` numbers: `padding="post"` adds zeros at the end and `truncating="post"` cuts the end of long reviews.
`train_test_split(..., stratify=y_train)` keeps the same share of positive and negative reviews in both parts.
**Expected output:** `Training set: (20000, 200)`, `Validation set: (5000, 200)`, `Test set: (25000, 200)`, and a review row of numbers (with zeros at the end for short reviews).
**Common errors:** a shape other than 200 columns means `MAX_LEN` was changed after an earlier cell ran: rerun from Step 4 in order.

---
# Step 7: Build the LSTM Model

**What:** define the neural network layer by layer.
**Why:** this is the model that will learn to separate positive from negative reviews.

| Layer | Purpose |
|---|---|
| `Embedding(10000, 64)` | Turns each word number into a learned 64-number vector. Similar words (like "great" and "excellent") end up with similar vectors. `mask_zero=True` makes the model ignore padding. |
| `LSTM(64)` | Reads the 200 word vectors in order and produces one 64-number summary of the whole review |
| `Dropout(0.3)` | Randomly switches off 30% of units during training to reduce overfitting |
| `Dense(32, relu)` | A small layer that combines the summary into higher-level features |
| `Dense(1, sigmoid)` | Outputs a single number between 0 and 1 = probability that the review is **positive** |

Loss function: **binary cross-entropy** (the standard loss for two classes). Optimizer: **Adam** (adjusts the weights efficiently).

In [ ]:
keras.backend.clear_session()

model = keras.Sequential([
    keras.Input(shape=(MAX_LEN,)),
    layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM, mask_zero=True),
    layers.LSTM(LSTM_UNITS, dropout=0.3),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss="binary_crossentropy",
              metrics=["accuracy"])
model.summary()

**Code explained.** `Sequential` stacks layers in order. `keras.Input` states the input shape (200 numbers per review). `compile` chooses the loss, optimizer and the metric to display.
`model.summary()` prints a table of layers and the number of parameters (learnable numbers).
**Expected output:** a table with five layers. The embedding layer has `10000 x 64 = 640,000` parameters, and the final `Dense` layer outputs shape `(None, 1)`.
**Common errors:**
| Error | Fix |
|---|---|
| `ValueError: ... incompatible with the layer` | Input shape mismatch: make sure `MAX_LEN` and the padded data agree (rerun Steps 4-6) |
| Output shape shows `None` | Normal: `None` means "any batch size" |
| Kernel dies / out of memory | Reduce `BATCH_SIZE` to 32 or `LSTM_UNITS` to 32 |

---
# Step 8: Train the Model

**What:** show the training reviews to the model repeatedly (each full pass is an **epoch**) and let it adjust its weights.
**Why:** this is where learning happens. After every epoch we measure performance on the validation set.

Two helpful **callbacks**:
- `EarlyStopping`: stops training when validation loss has not improved for 2 epochs and restores the best weights, which prevents overfitting and saves time.
- `ModelCheckpoint`: saves the best model to a file.

**Time:** on a typical CPU this can take roughly 1 to 3 minutes per epoch (it varies a lot between computers); on a GPU it is much faster.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("lstm_sentiment.keras", monitor="val_loss", save_best_only=True),
]

start = time.perf_counter()
history = model.fit(
    x_train, y_train_split,
    validation_data=(x_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=callbacks, verbose=1,
)
training_seconds = time.perf_counter() - start
print(f"\nTraining finished in {training_seconds/60:.1f} minutes, {len(history.history['loss'])} epochs run.")

**Code explained.** `model.fit` trains the network. `validation_data` is the held-out 5,000 reviews. `batch_size=64` means the weights are updated after every 64 reviews.
The `history` object stores loss and accuracy for each epoch so we can plot them.
**Expected output:** one progress bar per epoch showing `loss`, `accuracy`, `val_loss` and `val_accuracy`. Normally training accuracy keeps rising and validation accuracy rises
and then levels off. Early in training, accuracy near 50% means the model has not learned yet; this is normal for the first epoch.
**Common errors:**
| Problem | Fix |
|---|---|
| Accuracy stays near 50% for all epochs | Make sure the data and labels were prepared correctly (rerun Steps 4-6), try a few more epochs |
| `val_accuracy` falls while `accuracy` keeps rising | Overfitting. Early stopping handles it; you can also lower `LSTM_UNITS` or raise dropout |
| `ResourceExhaustedError` / out of memory | Lower `BATCH_SIZE` |
| Training is very slow | Use Google Colab with a GPU, or reduce `EPOCHS` |

---
# Step 9: Visualize Training

**What:** plot accuracy and loss for training and validation.
**Why:** the curves show whether the model is learning, underfitting or overfitting.

How to read them:
| What you see | Meaning |
|---|---|
| Both accuracy curves rise and stay close | Healthy learning |
| Training accuracy keeps rising, validation accuracy flat or falling | Overfitting (memorizing) |
| Validation loss starts to rise while training loss falls | Overfitting; the best epoch is where validation loss is lowest |
| Both curves low and flat | Underfitting (model too weak, too few epochs, or a data problem) |

In [ ]:
hist = pd.DataFrame(history.history)
hist.index = hist.index + 1          # epochs start at 1
print(hist.round(4))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist.index, hist["accuracy"], marker="o", label="Training")
ax[0].plot(hist.index, hist["val_accuracy"], marker="o", label="Validation")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(hist.index, hist["loss"], marker="o", label="Training")
ax[1].plot(hist.index, hist["val_loss"], marker="o", label="Validation")
ax[1].set_title("Loss"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

**Code explained.** The history dictionary becomes a table with one row per epoch. Two line charts compare training and validation values. The figure is saved as `training_curves.png` for your report.
**Expected output:** a table of per-epoch values and two line charts. The exact numbers depend on your run.
**Common errors:** `KeyError: 'accuracy'` appears if the training cell was not run first: run Step 8 before this step.

---
# Step 10: Evaluate on the Test Set

**What:** measure how well the final model performs on the 25,000 test reviews it has never seen.
**Why:** this gives an honest estimate of real-world performance.

Metrics (positive = the class we call "Positive"):
- **Accuracy** = correct predictions / all predictions
- **Precision** = of the reviews predicted Positive, how many really are positive: `TP / (TP + FP)`
- **Recall** = of the truly positive reviews, how many we found: `TP / (TP + FN)`
- **F1-score** = balance of precision and recall: `2 x P x R / (P + R)`
- **Confusion matrix** = table of correct and wrong predictions for each class (TN, FP, FN, TP)

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, batch_size=256, verbose=0)

probabilities = model.predict(x_test, batch_size=256, verbose=0).ravel()
y_pred = (probabilities >= 0.5).astype(int)           # above 0.5 means Positive

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "test_loss": float(test_loss),
}
print(pd.Series(metrics).round(4).to_string())
print("\n", classification_report(y_test, y_pred, target_names=["Negative", "Positive"], digits=4))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"])
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix (test set)")
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=150); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives={tn}, False Positives={fp}, False Negatives={fn}, True Positives={tp}")

**Code explained.** `model.predict` returns a probability for each review; we call it Positive if the probability is at least 0.5. Scikit-learn then compares these predictions with the true labels
to compute the metrics. The heatmap draws the confusion matrix: the diagonal cells are correct predictions, the off-diagonal cells are mistakes.
**Expected output:** a small table with accuracy, precision, recall, F1 and loss; a classification report per class; a 2x2 heatmap; and the four counts. The four counts add up to 25,000.
Because the test set is balanced, precision, recall and F1 should be close to the accuracy.
**Common errors:** `NameError: x_test is not defined` means an earlier cell was skipped: run the notebook from the top. A heatmap that looks all one colour usually means the model predicts only one class
(it did not learn): check the training curves.

---
# Step 11: Predict the Sentiment of New Reviews

**What:** turn any new text into the same number format used in training, then ask the model for its prediction.
**Why:** this is the purpose of the project: classify unseen reviews. The text must be encoded **exactly** like the training data (same dictionary, same offset of 3, same vocabulary limit, same padding),
otherwise predictions will be meaningless.

In [ ]:
import re

def encode_review(text):
    # lowercase, keep letters/digits/apostrophes, convert each word to its number
    words = re.findall(r"[a-z0-9']+", text.lower())
    ids = [1]                                    # 1 = start-of-review marker
    for w in words:
        number = word_index.get(w)
        if number is None or number + 3 >= VOCAB_SIZE:
            ids.append(2)                        # 2 = unknown word
        else:
            ids.append(number + 3)               # Keras shifts word numbers by 3
    return pad_reviews([ids])

def predict_sentiment(text):
    if not text or not text.strip():
        raise ValueError("Please provide a non-empty review.")
    p_positive = float(model.predict(encode_review(text), verbose=0)[0][0])
    label = LABELS[int(p_positive >= 0.5)]
    confidence = p_positive if p_positive >= 0.5 else 1 - p_positive
    return label, confidence

new_reviews = [
    "An absolutely wonderful film with brilliant acting and a touching story.",
    "Boring, predictable and far too long. I wanted my money back.",
    "The plot was weak, but the soundtrack and visuals were stunning.",
    "I did not enjoy this movie at all.",
    "One of the best movies I have seen this year, I loved every minute.",
]
for review in new_reviews:
    label, conf = predict_sentiment(review)
    print(f"{label:8s} ({conf:.1%})  {review}")

**Code explained.** `encode_review` lower-cases the text, splits it into words, looks each word up in the dictionary and applies the same +3 shift and vocabulary limit as the dataset.
`predict_sentiment` returns the label and a confidence value. The model's output is the probability of "Positive"; if it is below 0.5 the label is "Negative" and the confidence is `1 - probability`.
**Expected output:** one line per review with a label and a confidence percentage. Clearly positive or negative sentences are usually classified correctly; mixed or sarcastic reviews (like the third one) are harder
and the model may be uncertain or wrong. Judge the results yourself and report what you actually observe.
**Try your own:** change the text in `new_reviews`, or run `predict_sentiment("your review here")` in a new cell.
**Common errors:** `ValueError: Please provide a non-empty review.` means the text was empty. A `KeyError` is not expected because unknown words become the unknown marker.

---
# Step 12: Look at the Model's Mistakes (Error Analysis)

**What:** read a few test reviews that the model classified wrongly.
**Why:** metrics only say *how often* the model is wrong; reading mistakes shows *why* (sarcasm, mixed opinions, long reviews cut at 200 words, unknown words). This is valuable for your discussion section.

In [ ]:
wrong = np.where(y_pred != y_test)[0]
print(f"Misclassified test reviews: {len(wrong)} of {len(y_test)} ({len(wrong)/len(y_test):.1%})\n")

rng = np.random.default_rng(SEED)
for i in rng.choice(wrong, size=min(4, len(wrong)), replace=False):
    print(f"True: {LABELS[int(y_test[i])]:8s} | Predicted: {LABELS[int(y_pred[i])]:8s} | P(positive)={probabilities[i]:.2f}")
    print("   ", decode_review(x_test_raw[i])[:350], "...\n")

**Code explained.** `np.where` finds the positions where prediction and truth differ. We print four random mistakes with their decoded text. `<UNK>` marks words outside the 10,000-word vocabulary.
**Expected output:** the number of mistakes and four examples. When you read them, look for patterns and mention them in your report (only what you actually see).
**Common errors:** none expected. If `wrong` is empty (it will not be in practice), `rng.choice` would fail.

---
# Step 13: Save the Model and Results

**What:** store the model and the metrics in files.
**Why:** so you can reuse the trained model without retraining, and attach the results to your report or GitHub repository.

In [ ]:
model.save("lstm_sentiment.keras")
with open("metrics.json", "w") as f:
    json.dump({k: float(v) for k, v in metrics.items()}, f, indent=2)
print("Saved: lstm_sentiment.keras, metrics.json, training_curves.png, confusion_matrix.png")

# How to load the model later (without retraining):
reloaded = keras.models.load_model("lstm_sentiment.keras")
check = float(reloaded.predict(encode_review("A fantastic and moving movie."), verbose=0)[0][0])
print("Reloaded model works, P(positive) for a test sentence =", round(check, 3))

**Code explained.** `model.save` writes the architecture and weights to one file. `json.dump` stores the metrics. `load_model` shows how to reload the model, and one prediction proves it still works.
**Expected output:** the list of saved files and one probability.
**Common errors:** on Colab, files vanish after the session ends, so download them. If saving fails with a permission error on Windows, move the project out of a protected folder such as `C:\Program Files`.

---
# Step 14: Results Summary, Discussion and Conclusion

**Fill in this table with your own numbers (from Step 10):**

| Metric | Value from my run |
|---|---|
| Test accuracy | [fill in] |
| Precision | [fill in] |
| Recall | [fill in] |
| F1-score | [fill in] |
| Epochs run / training time | [fill in] |

**Discussion questions to answer in your report (use your own evidence):**
1. Does the training/validation curve show overfitting? At which epoch was validation loss lowest?
2. Are precision and recall similar? What does the confusion matrix tell you about which class is easier?
3. What patterns did you notice in the misclassified reviews?
4. How did the model handle your own reviews, especially mixed or negated sentences?

**Conclusion (template; edit after you have your results).** An LSTM network was trained on the IMDB movie review dataset to classify reviews as positive or negative.
Reviews were converted to integer sequences, limited to a 10,000-word vocabulary and padded to 200 words. The model's performance on 25,000 unseen test reviews was [your accuracy and F1].
LSTM networks are well suited to this task because they read text in order and keep context in their memory.

**Limitations**
- Only the first 200 words of each review are used.
- Rare words outside the 10,000-word vocabulary are treated as unknown.
- Training data are English movie reviews, so the model may not work well for other topics, languages or sarcasm.
- A single run with one random seed; results may vary slightly between runs.

**Future improvements**
- Bidirectional LSTM or GRU; stacked LSTM layers.
- Pre-trained word embeddings such as GloVe.
- Hyperparameter tuning (units, dropout, learning rate, `MAX_LEN`).
- Transformer models (such as BERT) for higher accuracy.
- A small web interface (Streamlit or Flask) for typing reviews.

---
# Step 15: Publish the Project on GitHub

**What / why:** Git tracks changes to your files; GitHub hosts the repository so your instructor can see your work.

1. Install Git for Windows (git-scm.com) and create a free account at github.com. In PowerShell: `git --version`.
2. Tell Git who you are (once): 
```powershell
git config --global user.name "Your Name"
git config --global user.email "you@example.com"
```
3. On github.com click **New repository**, name it `lstm-movie-review-sentiment`, keep it empty (no README), and create it.
4. In your project folder (VS Code terminal):
```powershell
git init
git add .
git commit -m "LSTM sentiment analysis of IMDB movie reviews"
git branch -M main
git remote add origin https://github.com/<your-username>/lstm-movie-review-sentiment.git
git push -u origin main
```
5. Refresh the GitHub page; your files should appear. Take a screenshot if your assignment asks for one.

**Common errors**
| Error | Fix |
|---|---|
| `git is not recognized` | Install Git and reopen VS Code |
| `Authentication failed` / password rejected | GitHub no longer accepts account passwords for Git. Sign in through the browser window Git opens, or use a personal access token |
| `remote origin already exists` | `git remote set-url origin <URL>` |
| `failed to push some refs` | The repository was not empty: `git pull origin main --allow-unrelated-histories`, then push again |
| Model file too large | The `.gitignore` in this project skips `*.keras`; do not upload large files unless required |

**Do not commit** your virtual environment (`.venv`) or any secrets. The supplied `.gitignore` already excludes `.venv`.

## References
- Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation, 9*(8), 1735-1780.
- Maas, A. L., Daly, R. E., Pham, P. T., Huang, D., Ng, A. Y., & Potts, C. (2011). Learning word vectors for sentiment analysis. In *Proceedings of the 49th Annual Meeting of the Association for Computational Linguistics* (pp. 142-150).
- Chollet, F. et al. (2015). *Keras*. https://keras.io
- Abadi, M. et al. (2016). TensorFlow: A system for large-scale machine learning. In *OSDI '16* (pp. 265-283).
- Pedregosa, F. et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825-2830.

*(Verify each reference against the original source before submitting.)*